# Exercises

In this section we have two exercises:
1. Implement the polynomial kernel.
2. Implement the multiclass C-SVM.

## Polynomial kernel

You need to extend the ``build_kernel`` function and implement the polynomial kernel if the ``kernel_type`` is set to 'poly'. The equation that needs to be implemented:
\begin{equation}
K=(X^{T}*Y)^{d}.
\end{equation}

In [1]:
def build_kernel(data_set, kernel_type='linear'):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    elif kernel_type == 'poly':
        d = 2
        kernel = kernel ** d  # put your code here
    return kernel

## Implement a multiclass C-SVM

Use the classification method that we used in notebook 7.3 and IRIS dataset to build a multiclass C-SVM classifier. Most implementation is about a function that will return the proper data set that need to be used for the prediction. You need to implement:
- ``choose_set_for_label``
- ``get_labels_count``

In [2]:
def choose_set_for_label(data_set, label):
    # your code here
    features = data_set[:, :-1]
    labels = data_set[:, -1].astype(int)
    train_data_set, test_data_set, train_labels, test_labels = train_test_split(
        features, labels, test_size=0.2, random_state=15)
    train_labels = np.where(train_labels == label, 1, -1).reshape(-1, 1)
    test_labels = np.where(test_labels == label, 1, -1).reshape(-1, 1)
    return train_data_set, test_data_set, train_labels, test_labels

In [3]:
def get_labels_count(data_set):
    labels_count = len(np.unique(data_set[:, -1]))
    return labels_count

In [4]:
# Use the code that we have implemented earlier:

In [5]:
def train(train_data_set, train_labels, kernel_type='linear', C=10, threshold=1e-5):
    kernel = build_kernel(train_data_set, kernel_type=kernel_type)

    P = train_labels * train_labels.transpose() * kernel
    q = -np.ones((objects_count, 1))
    G = np.concatenate((np.eye(objects_count), -np.eye(objects_count)))
    h = np.concatenate((C * np.ones((objects_count, 1)), np.zeros((objects_count, 1))))

    A = train_labels.reshape(1, objects_count)
    A = A.astype(float)
    b = 0.0

    sol = cvxopt.solvers.qp(cvxopt.matrix(P), cvxopt.matrix(q), cvxopt.matrix(G), cvxopt.matrix(h), cvxopt.matrix(A), cvxopt.matrix(b))

    lambdas = np.array(sol['x'])

    support_vectors_id = np.where(lambdas > threshold)[0]
    vector_number = len(support_vectors_id)
    support_vectors = train_data_set[support_vectors_id, :]

    lambdas = lambdas[support_vectors_id]
    targets = train_labels[support_vectors_id]

    b = np.sum(targets)
    for n in range(vector_number):
        b -= np.sum(lambdas * targets * np.reshape(kernel[support_vectors_id[n], support_vectors_id], (vector_number, 1)))
    b /= len(lambdas)

    return lambdas, support_vectors, support_vectors_id, b, targets, vector_number

def build_kernel(data_set, kernel_type='linear'):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    elif kernel_type == 'poly':
        d = 2
        kernel = kernel ** d
    return kernel

def classify_rbf(test_data_set, train_data_set, lambdas, targets, b, vector_number, support_vectors, support_vectors_id, use_sign=True):
    kernel = np.dot(test_data_set, support_vectors.T)
    sigma = 1.0
    #K = np.dot(test_data_set, support_vectors.T)
    #kernel = build_kernel(train_data_set, kernel_type='rbf')
    c = (1. / sigma * np.sum(test_data_set ** 2, axis=1) * np.ones((1, np.shape(test_data_set)[0]))).T
    c = np.dot(c, np.ones((1, np.shape(kernel)[1])))
    #aa = np.dot((np.diag(K)*np.ones((1,len(test_data_set)))).T[support_vectors_id], np.ones((1, np.shape(K)[0]))).T
    sv = (np.diag(np.dot(train_data_set, train_data_set.T))*np.ones((1,len(train_data_set)))).T[support_vectors_id]
    aa = np.dot(sv,np.ones((1,np.shape(kernel)[0]))).T
    kernel = kernel - 0.5 * c - 0.5 * aa
    kernel = np.exp(kernel / (2. * sigma ** 2))

    y = np.zeros((np.shape(test_data_set)[0], 1))
    for j in range(np.shape(test_data_set)[0]):
        for i in range(vector_number):
            y[j] += lambdas[i] * targets[i] * kernel[j, i]
        y[j] += b
    if use_sign:
        return np.sign(y)
    return y

In [6]:
# modify this part 
from sklearn.datasets import load_iris
import numpy as np
from sklearn.model_selection import train_test_split
!pip install cvxopt 
import cvxopt
from sklearn.metrics import accuracy_score

iris = load_iris()
data_set = np.column_stack([iris.data, iris.target])
labels_count = get_labels_count(data_set)

_, test_data_set_orig, _, test_labels_orig = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=15)

scores = np.zeros((len(test_data_set_orig), labels_count))
for label in range(labels_count):
    train_data_set, test_data_set, train_labels, test_labels = choose_set_for_label(data_set, label)
    objects_count = len(train_labels)
    lambdas, support_vectors, support_vectors_id, b, targets, vector_number = train(train_data_set, train_labels, kernel_type='rbf')
    predicted = classify_rbf(test_data_set, train_data_set, lambdas, targets, b, vector_number, support_vectors, support_vectors_id, use_sign=False)
    scores[:, label] = predicted.flatten()

predicted = list(np.argmax(scores, axis=1))
print(accuracy_score(predicted, test_labels_orig))

     pcost       dcost       gap    pres   dres
 0:  1.3324e+02 -1.9748e+03  4e+03  2e-01  3e-15
 1:  8.6383e+01 -2.0657e+02  3e+02  8e-03  2e-15
 2:  1.1785e+01 -2.6208e+01  4e+01  5e-16  3e-15
 3: -6.8044e-01 -5.9075e+00  5e+00  2e-16  1e-15
 4: -1.8036e+00 -3.4676e+00  2e+00  3e-16  6e-16
 5: -2.2743e+00 -2.7856e+00  5e-01  2e-16  4e-16
 6: -2.3977e+00 -2.6013e+00  2e-01  2e-16  4e-16
 7: -2.4790e+00 -2.5651e+00  9e-02  4e-16  4e-16
 8: -2.5016e+00 -2.5138e+00  1e-02  3e-16  4e-16
 9: -2.5059e+00 -2.5061e+00  2e-04  3e-16  5e-16
10: -2.5060e+00 -2.5060e+00  5e-06  2e-16  6e-16
11: -2.5060e+00 -2.5060e+00  3e-07  7e-16  5e-16
Optimal solution found.
     pcost       dcost       gap    pres   dres
 0:  1.7477e+02 -4.0339e+03  7e+03  2e-01  5e-15
 1:  9.0703e+01 -5.1390e+02  7e+02  1e-02  5e-15
 2: -3.6896e+01 -2.1569e+02  2e+02  3e-03  5e-15
 3: -7.6419e+01 -1.3538e+02  6e+01  7e-04  4e-15
 4: -9.2336e+01 -1.2094e+02  3e+01  2e-04  5e-15
 5: -9.7622e+01 -1.0927e+02  1e+01  4e-05  4e-1